# 10 - Human-in-the-Loop Exception Workflow

Metatate decisions are operational. Safe requests can proceed, conditional requests should become reviewer-ready exception packets, and denied requests should remain blocked.

This notebook uses the same `human_exception_workflow` package as the command-line runner.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Review the workflow requests


In [ ]:
from human_exception_workflow import DEFAULT_REQUESTS, print_summary, run_workflow

pd.DataFrame(DEFAULT_REQUESTS)[
    ["request_id", "kind", "title", "operation", "intended_use", "owner"]
]


## Run the exception workflow through Metatate


In [ ]:
workflow_run = run_workflow(client)
report = workflow_run.to_dict()
print_summary(workflow_run)

pd.DataFrame(report["items"])[
    ["request_id", "title", "decision", "status", "evidence_id"]
]


## Inspect the conditional exception packet


In [ ]:
conditional_item = next(item for item in report["items"] if item["request_id"] == "req-002")
print(json.dumps(conditional_item["packet"], indent=2))


## Reviewer decision and resumed workflow


In [ ]:
review = conditional_item["review"]
resume_payload = conditional_item["resume_payload"]

print("Reviewer decision")
print(json.dumps(review, indent=2))
print("\nResume payload")
print(json.dumps(resume_payload, indent=2))


## Blocked requests do not resume


In [ ]:
blocked_item = next(item for item in report["items"] if item["request_id"] == "req-003")
print(json.dumps({
    "request_id": blocked_item["request_id"],
    "decision": blocked_item["decision"],
    "status": blocked_item["status"],
    "evidence_id": blocked_item["evidence_id"],
    "rationale": blocked_item["packet"]["rationale"],
    "resume_payload": blocked_item["resume_payload"],
}, indent=2))


## Run the same workflow from a terminal


In [ ]:
print("scripts/run_human_exception_workflow.sh")
print("scripts/run_human_exception_workflow_acceptance.sh")


This is the bridge from agent output to governance operations. Conditional decisions become controlled review work; denied decisions remain blocked instead of becoming informal overrides.
